# SpottingSalmon
Model prediction scipt
- To use final YOLO model to generate fish counts from unseen videos.

_**!Important! Make sure you are connected to a GPU cluster i.e. salmonGPU **_

### Step 1: Load best model

In [0]:
%pip install ultralytics

In [0]:
import os
from ultralytics import YOLO

model = YOLO("/Users/rachel.lennon@defra.gov.uk/fish_yolo_experiment/fish_yolo_m4/weights/best.pt")

### Step 2: Customise tracker
Generate a tracker so that the same fish in multiple frames is identified as one fish.

In [0]:
yaml_content = """
tracker_type: bytetrack 
track_high_thresh: 0.25 # First-stage match threshold
track_low_thresh: 0.1 # Second-stage threshold for low-score matches
new_track_thresh: 0.25 # Start a new track if no match > this
track_buffer: 1 # Frames to keep lost tracks alive; higher handles occlusion, increases ID switches risk
match_thresh: 0.8 # (float) Association similarity threshold (IoU/cost); tune with detector quality
fuse_score: True # (bool) Fuse detection score with motion/IoU for matching; stabilizes weak detections
max_lost: 10 
"""

yaml_path = "/dbfs/FileStore/rachlenn/bytetrack_fish.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"Custom ByteTrack YAML saved at: {yaml_path}")


In [0]:
import yaml

with open("/dbfs/FileStore/rachlenn/bytetrack_fish.yaml") as f:
    config = yaml.safe_load(f)
print(config)


### Step 3: Use model and tracker to predict counts on unseen videos

In [0]:
import os
import pandas as pd
from ultralytics import YOLO

video_dir = "/dbfs/mnt/lab/unrestricted/rachel.lennon@defra.gov.uk/videos/unseen/"
output_summary_file = "/dbfs/mnt/lab/unrestricted/rachel.lennon@defra.gov.uk/videos/unseen/fish_tracking_summary.csv"

# Initialize summary
results_summary = []

# Process each video
for video in os.listdir(video_dir):
    if not video.endswith(".mp4"):
        continue

    path = os.path.join(video_dir, video)
    print(f"Processing: {video}")

    # Run ByteTrack tracking
    results = model.track(
        source=path,
        conf=0.15,  # lower to catch small/fast fish
        tracker="/dbfs/FileStore/rachlenn/bytetrack_fish.yaml",
        save=True,
        device=0
    )

    # Collect unique fish IDs
    fish_ids = set()
    for r in results:
        if hasattr(r.boxes, "id") and r.boxes.id is not None:
            ids = r.boxes.id.cpu().numpy().tolist()
            fish_ids.update(ids)

    results_summary.append({
        "video": video,
        "fish_count": len(fish_ids)
    })

# Save summary to CSV
df_summary = pd.DataFrame(results_summary)
df_summary.to_csv(output_summary_file, index=False)

print(f"Summary saved to: {output_summary_file}")
print(df_summary)


In [0]:
import pandas as pd

# Read CSV with pandas
df = pd.read_csv("/dbfs/mnt/lab/unrestricted/rachel.lennon@defra.gov.uk/videos/unseen/fish_tracking_summary.csv")

# Display in Databricks notebook
display(df)
